# 2. Air Quality Data — Download

## Data source

**[AQICN COVID-19 Data Platform](https://aqicn.org/data-platform/covid19/)**

The World Air Quality Index (WAQI) project provides historical air quality data
aggregated from monitoring stations worldwide. During the COVID-19 pandemic, they
compiled a special dataset with **daily city-level** air quality measurements for
research purposes.

## What this notebook does

This notebook **only downloads and saves the raw data**. No cleaning or
transformation is done here — that happens in notebook 3.

We download quarterly CSV files from the AQICN platform for **2020Q1 through 2022Q4**
(12 quarters = 3 years), filter to our 126 target cities across 9 European
countries, and save the combined raw dataset.

## Why this data?

This is our **outcome variable** data. We want to measure how city-level air quality
responded to COVID-19 restrictions. The AQICN dataset is ideal because:

- **Daily frequency**: allows us to track rapid changes around lockdown dates
- **City-level granularity**: matches our cross-sectional unit of analysis
- **Multiple pollutants**: lets us check which pollutants respond most to mobility restrictions
- **Weather data included**: provides built-in confounders for our regressions

## Data structure

Each quarterly CSV from AQICN contains **daily observations in long format**.
Each row is one city × date × species combination:

| Column | Description |
|--------|-------------|
| `Date` | Date of measurement (YYYY-MM-DD) |
| `Country` | 2-letter ISO country code (e.g., DE, FR, IT) |
| `City` | City name in English (e.g., Berlin, Paris, Rome) |
| `Specie` | Name of the pollutant or weather variable measured |
| `count` | Number of monitoring station readings aggregated that day |
| `min` | Minimum value across stations |
| `max` | Maximum value across stations |
| `median` | **Median across stations — this is our primary measurement** |
| `variance` | Variance across stations (useful for data quality assessment) |

The `Specie` column contains both **pollutants** and **weather variables**:

### Pollutants (outcome variables)

| Species | Full name | Why it matters |
|---------|-----------|----------------|
| `no2` | Nitrogen dioxide | **Primary outcome.** Directly emitted by vehicles; most responsive to traffic changes |
| `pm25` | Fine particulate matter (PM2.5) | Health-relevant; partly from traffic, partly from heating/industry |
| `pm10` | Coarse particulate matter | Includes dust and construction; less traffic-specific |
| `o3` | Ground-level ozone | Formed via NOx chemistry; may *increase* during lockdowns (see report) |
| `co` | Carbon monoxide | Vehicle exhaust; less commonly monitored |
| `so2` | Sulfur dioxide | Mainly from industry/power plants; less traffic-related |

### Weather variables (confounders/controls)

| Species | Description |
|---------|-------------|
| `temperature` | Air temperature (°C) |
| `humidity` | Relative humidity (%) |
| `pressure` | Atmospheric pressure (hPa) |
| `wind speed` | Wind speed (m/s) — disperses pollutants |
| `wind gust` | Maximum wind gust |
| `dew` | Dew point temperature |
| `precipitation` | Rainfall amount |

## API access

The AQICN COVID data platform provides pre-compiled quarterly CSVs via a
report-specific URL. The URL contains a personal access token. If the download
fails, you may need to register at https://aqicn.org/data-platform/covid19/
and generate your own report URL.

In [ ]:
import pandas as pd
import os
import time

## 2.1 Define target cities and download parameters

We load the city-to-country mapping created in notebook 1 to know which cities
to keep. We download 12 quarters (2020Q1–2022Q4) covering the full pandemic period
from pre-lockdown baseline through reopening.

In [ ]:
# Load city list from notebook 1 output
df_map = pd.read_csv('../data/clean/city_country_mapping.csv')
target_cities = set(df_map['aqicn_city'].tolist())
target_countries = set(df_map['aqicn_country'].tolist())

print(f'Target: {len(target_cities)} cities across {len(target_countries)} countries')

# Quarterly periods to download
periods = [
    '2020Q1', '2020Q2', '2020Q3', '2020Q4',
    '2021Q1', '2021Q2', '2021Q3', '2021Q4',
    '2022Q1', '2022Q2', '2022Q3', '2022Q4',
]

# AQICN report URL with access token
# This token is tied to a personal AQICN account.
# If it expires, register at https://aqicn.org/data-platform/covid19/
AQICN_TOKEN = '50428-9b6996dd'
BASE_URL = f'https://aqicn.org/data-platform/covid19/report/{AQICN_TOKEN}/{{}}'

print(f'Periods to download: {len(periods)} quarters ({periods[0]} to {periods[-1]})')

## 2.2 Download quarterly CSVs

We download each quarter, immediately filter to our target countries and cities
to reduce memory usage, and append to a list. A 1-second pause between requests
avoids overloading the AQICN server.

If the raw file already exists locally, we skip the download.

In [ ]:
raw_output_path = '../data/raw/aqicn_all_cities_raw.csv'

if os.path.exists(raw_output_path):
    print(f'Raw data already exists at {raw_output_path}, skipping download.')
    print('Delete the file and re-run this cell to force a fresh download.')
    df_all = pd.read_csv(raw_output_path)
else:
    all_data = []

    print('Downloading AQICN quarterly CSVs...')
    print('=' * 60)

    for period in periods:
        url = BASE_URL.format(period)
        print(f'  {period}...', end=' ')

        try:
            # AQICN CSVs have 4 lines of header text before the actual CSV data
            df_temp = pd.read_csv(url, skiprows=4)

            # Filter to target countries first (fast), then target cities
            df_temp = df_temp[df_temp['Country'].isin(target_countries)]
            df_temp = df_temp[df_temp['City'].isin(target_cities)]

            n_cities = df_temp['City'].nunique()
            n_rows = len(df_temp)
            print(f'{n_rows:,} rows, {n_cities} cities')

            if not df_temp.empty:
                all_data.append(df_temp)

            time.sleep(1)  # be nice to the server

        except Exception as e:
            print(f'FAILED: {e}')

    print('=' * 60)

    # Combine all quarters and sort
    # The quarterly CSVs may have inconsistent internal ordering, so we sort
    # by Country → City → Specie → Date to ensure a clean, browsable output.
    df_all = pd.concat(all_data, ignore_index=True)
    df_all['Date'] = pd.to_datetime(df_all['Date'])
    df_all = df_all.sort_values(['Country', 'City', 'Specie', 'Date']).reset_index(drop=True)

    print(f'\nTotal: {len(df_all):,} rows')
    print(f'Cities: {df_all["City"].nunique()}')
    print(f'Date range: {df_all["Date"].min().date()} to {df_all["Date"].max().date()}')

    # Save raw data
    df_all.to_csv(raw_output_path, index=False)
    print(f'\nSaved raw data to: {raw_output_path}')

## 2.3 Quick sanity check

Before moving to the cleaning notebook, we do a quick sanity check to make sure
the download captured what we expected.

In [ ]:
print(f'Shape: {df_all.shape}')
print(f'\nColumns: {df_all.columns.tolist()}')
print(f'\nSpecies (pollutants + weather): {sorted(df_all["Specie"].unique())}')
print(f'\nCountries: {sorted(df_all["Country"].unique())}')
print(f'\nCities per country:')
print(df_all.groupby('Country')['City'].nunique().to_string())

In [ ]:
# Check which target cities are MISSING from the downloaded data
downloaded_cities = set(df_all['City'].unique())
missing_cities = target_cities - downloaded_cities

if missing_cities:
    print(f'WARNING: {len(missing_cities)} target cities not found in AQICN data:')
    for c in sorted(missing_cities):
        country = df_map[df_map['aqicn_city'] == c]['aqicn_country'].iloc[0]
        print(f'  {country}: {c}')
else:
    print('All 126 target cities found in the downloaded data.')

In [ ]:
# Preview the raw data
df_all.head(10)